In [1]:
# Upgrade PyTorch to a unified CUDA 12.1 version
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install and upgrade all Hugging Face and evaluation libraries together
!pip install --upgrade transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score bert_score nltk

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
import gc
import torch
import os
import json
import glob
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
def migrate_essay_data(folder_path):
    all_essay_records = []
    search_pattern = os.path.join(folder_path, "*_essay.jsonl")
    essay_files = glob.glob(search_pattern)
    for file_path in essay_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        record = json.loads(line)
                        if record.get("type") == "essay" and record.get("model_answer"):
                            all_essay_records.append(record)
                    except: continue
    return all_essay_records

In [ ]:
def preprocess_function(examples):
    inputs = [f"summarize technical info: {c}" for c in examples["context"]]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LENGTH, truncation=True)

    targets = [f"Scenario: {s} Question: {q} Answer: {a}" 
               for s, q, a in zip(examples["scenario_context"], examples["question"], examples["model_answer"])]
    
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs 

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
MODEL_NAME = "google/flan-t5-base"  
DATA_FOLDER = "/kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions"
OUTPUT_DIR = "./outputs/STABLE_T5"

BATCH_SIZE = 4
GRADIENT_ACC_STEPS = 8 
MAX_SOURCE_LENGTH = 384
MAX_TARGET_LENGTH = 256

gc.collect()
torch.cuda.empty_cache()

raw_essay_data = migrate_essay_data(DATA_FOLDER)
dataset = Dataset.from_list(raw_essay_data)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    device_map="auto",
    torch_dtype=torch.float32 
)

peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q", "v", "wi_0", "wi_1", "wo"], 
    lora_dropout=0.1, 
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, peft_config)

In [ ]:
# ==========================================
# 4. TRAINING ARGUMENTS
# ==========================================
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=50,               
    logging_steps=5,
    learning_rate=2e-4,         
    num_train_epochs=3,          
    max_grad_norm=1.0,           
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    fp16=False,                  
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    optim="adamw_torch",         
    report_to="none"
)

In [2]:
rouge_metric = evaluate.load("rouge")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, 
        model=model, 
        padding=True,
        label_pad_token_id=-100  
    ),
    compute_metrics=compute_metrics
)

trainer.train()

Map:   0%|          | 0/2209 [00:00<?, ? examples/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
50,23.001039,2.581299,0.148577,0.021688,0.111961,0.111980
100,21.001500,2.358978,0.317995,0.074767,0.202493,0.202460
150,20.221468,2.305382,0.341174,0.089076,0.217431,0.217262
189,20.268845,2.295820,0.345763,0.090698,0.219300,0.219131


TrainOutput(global_step=189, training_loss=21.60246224378152, metrics={'train_runtime': 2603.5371, 'train_samples_per_second': 2.291, 'train_steps_per_second': 0.073, 'total_flos': 3117883879858176.0, 'train_loss': 21.60246224378152, 'epoch': 3.0})

In [5]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

BASE_MODEL_NAME = "google/flan-t5-base"
LORA_DIR = "/kaggle/working/outputs/STABLE_T5/checkpoint-189"

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME, 
    device_map="auto",
    torch_dtype=torch.float32
)

print("Applying custom LoRA weights...")
model = PeftModel.from_pretrained(base_model, LORA_DIR)

def ask_my_model(context_text):
    prompt = f"summarize technical info: {context_text}"
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=384, truncation=True).to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_length=256,
        num_beams=4,         
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_context = """
A DNS server translates human-readable domain names (like www.google.com) into IP addresses. 
If a DNS server goes down, users will experience 'server not found' errors unless they have 
the IP address cached locally.
"""

print("\n--- MODEL GENERATION ---")
response = ask_my_model(sample_context)
print(response)

Loading base model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Applying custom LoRA weights...

--- MODEL GENERATION ---
Scenario: Your website's DNS server is down, and you're experiencing 'server not found' errors. Question: Propose a solution to resolve this issue. Explain the tradeoffs involved. Answer: Implement a DNS server that caches the IP address locally. This will ensure that users don't experience 'server not found' errors unless they have the IP address cached locally.
